# mlmodels x datasets -- end-to-end integration spike

**Purpose.** Combine both bounded contexts (BCs) in one runnable place and prove they
compose: load the real neutron-stars data (`datasets` BC), train a *simple injected
NN* through the `mlmodels` application port, then persist / find / load / **materialize**
a live surrogate that predicts -- exercising every seam the persistence API (L2b) built.

**This is a spike, not production code.** It wires `domain` / `application` /
`infrastructure` directly (a notebook is a runtime experiment surface, like a delivery
edge), uses the real dataset (allowed at runtime, never in tests), and trains a tiny
model on CPU. It exists to surface BC-sync bugs before committing to the managed
training phase (L2c).

**What it checks**
1. `datasets` BC load -> tensors (the DIP data seam, done by hand here).
2. A simple multi-feature NN that declares its own `MODEL_NAME` / `MODEL_VERSION`.
3. An injected `SaveTrainedRunFn` adapter that actually **fits** -- driven through
   `handle_train_run` (the same application seam `test_e2e.py` uses).
4. Read side (`find_run_summary`), load side (`load_trained_run`), and
   `materialize_model` with both reload guards passing -> a **prediction + test RMSE**.
5. Diagnostics that deliberately probe three BC-sync fault lines (guard firing,
   the fork-2 checkpoint-format mismatch, and the untrained shipped facade).

## 0. Config / env seam

`get_settings()` reads `.env` **relative to the process working directory**, so a
notebook launched under `notebooks/` would miss the project `.env`, and the built-in
defaults point at `/var/...` (filesystem root) rather than the project tree. So we set
the `SURROGATE_MODELS__*` variables explicitly **before importing the package** (OS env
is the highest-priority source). This makes the notebook run from any working directory.

In [ ]:
import os
from pathlib import Path


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("pyproject.toml not found above cwd")


ROOT = project_root()
VAR = ROOT / "var" / "data" / "surrogate_models"
CKPT_DIR = VAR / "checkpoints" / "nb-e2e"

os.environ["SURROGATE_MODELS__DATASETS__PATH"] = str(VAR / "datasets")
os.environ["SURROGATE_MODELS__DATASETS__NEUTRON_STARS_SOURCE"] = str(
    ROOT / "data" / "neutron-stars" / "neutron-stars.dat"
)
os.environ["SURROGATE_MODELS__MLMODELS__CHECKPOINT_DIR"] = str(CKPT_DIR)
os.environ["SURROGATE_MODELS__LOGGING__FILE"] = str(VAR / "log" / "surrogate_models.log")
print("project root:", ROOT)
print("checkpoint dir:", CKPT_DIR)

In [ ]:
from functools import partial
from typing import Any

import lightning
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Public facades (top-level package API).
from surrogate_models import TrainRun, load_neutron_stars, train_run

# mlmodels application seam (the injectable ports + handlers).
from surrogate_models.mlmodels.application import (
    GetRunSummary,
    handle_get_run_summary,
    handle_train_run,
)
# domain value objects / smart constructors we assemble by hand in the spike.
from surrogate_models.mlmodels.domain import Checkpoint, ConfiguredRun, make_model_identity
# infrastructure adapters the persistence API exposes.
from surrogate_models.mlmodels.infrastructure import (
    RunManifest,
    find_run_summary,
    load_trained_run,
    materialize_model,
    read_run_manifest,
    structural_fingerprint,
    write_run_manifest,
)
from surrogate_models.railway_adts import fmap_error, safe

torch.manual_seed(0)
_ = lightning.seed_everything(0, verbose=False)

## 1. `datasets` BC -- load real data and shape it into tensors

`load_neutron_stars()` is the datasets BC's one-call public loader (get-or-build with
read-through caching). We subsample, pick two physical features, standardize, and hold
out a test slice. Standardization matters: the raw columns span many orders of magnitude.

This hand-wiring **is** the DIP data seam L2c will formalize: data reaches `mlmodels`
as injected tensors, never by `mlmodels` importing `datasets`.

In [ ]:
df = load_neutron_stars()
print("full dataset:", df.shape)

FEATURES = ["rhoc", "Pc"]      # central density, central pressure
TARGET = "M"                   # gravitational mass (Msun)

sample = df[[*FEATURES, TARGET]].dropna().sample(n=3000, random_state=0)
x_raw = torch.tensor(sample[FEATURES].to_numpy(), dtype=torch.float32)
y_raw = torch.tensor(sample[[TARGET]].to_numpy(), dtype=torch.float32)

x_mean, x_std = x_raw.mean(0), x_raw.std(0)
y_mean, y_std = y_raw.mean(0), y_raw.std(0)
X = (x_raw - x_mean) / x_std
Y = (y_raw - y_mean) / y_std

N_TEST = 500
X_train, Y_train = X[:-N_TEST], Y[:-N_TEST]
X_test, Y_test = X[-N_TEST:], Y[-N_TEST:]
print("train:", tuple(X_train.shape), "test:", tuple(X_test.shape))

## 2. A simple NN that declares its own identity

A tiny `2 -> 8 -> 1` MLP (this is the "simple NN model" to inject). It follows the same
LightningModule contract as `test_e2e.py`'s `StubRegressor` -- `forward`,
`training_step` (MSE), `configure_optimizers` -- but is multi-feature, unlike the
shipped `SurrogateRegressor` (`1 -> 1`). It declares `MODEL_NAME` / `MODEL_VERSION`, the
identity a run records so `materialize_model` can prove a reload rebuilds the same model.

In [ ]:
class SimpleSurrogate(lightning.LightningModule):
    '''A 2->8->1 MLP surrogate for neutron-star mass -- notebook model only.'''

    MODEL_NAME = "ns-mlp-surrogate"
    MODEL_VERSION = "0.1.0"

    def __init__(self, n_features: int, learning_rate: float, optimizer_name: str) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 8), nn.ReLU(), nn.Linear(8, 1))
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer_name

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.net(features)
        return out

    def training_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        features, targets = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(features), targets)
        self.log("train_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        match self.optimizer_name:
            case "adam":
                return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
            case _:
                return torch.optim.SGD(self.parameters(), lr=self.learning_rate)


N_FEATURES = len(FEATURES)
IDENTITY = make_model_identity(
    SimpleSurrogate.MODEL_NAME, SimpleSurrogate.MODEL_VERSION
).unwrap()


def build_model(config: Any) -> SimpleSurrogate:
    '''Factory: build the surrogate FROM a run's certified config.'''
    return SimpleSurrogate(N_FEATURES, config.learning_rate, config.optimizer)

## 3. Inject a real training adapter (the `SaveTrainedRunFn` seam)

The shipped `save_trained_run` builds an **untrained** `1->1` model. To train a real
surrogate we supply our own adapter of the same port shape
`ConfiguredRun -> Result[Checkpoint]` -- exactly the injection `test_e2e.py` uses. It:

- builds the NN from the run's certified config,
- **fits** it with a Lightning `Trainer` over the injected tensors,
- writes a **bare** `state_dict` to `{run_id}.ckpt` (what `materialize_model` expects --
  see diagnostic 8b), and
- writes the `{run_id}.json` manifest carrying the model's identity + structural
  fingerprint, so the read / load / materialize paths line up.

In [ ]:
@safe(Exception, fmap_error(lambda cause: cause, code="TRAINING_FAILED"))
def real_save_trained_run(
    checkpoint_dir: Path, x: torch.Tensor, y: torch.Tensor, run: ConfiguredRun
) -> Checkpoint:
    '''Fit the surrogate for ``run`` and persist ckpt + manifest -- injected adapter.'''
    model = build_model(run.config)
    loader = DataLoader(
        TensorDataset(x, y), batch_size=run.config.batch_size, shuffle=True
    )
    trainer = lightning.Trainer(
        max_epochs=run.config.max_epochs,
        accelerator="cpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, loader)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    target = checkpoint_dir / f"{run.run_id}.ckpt"
    torch.save(model.state_dict(), target)  # BARE state_dict -- matches materialize
    write_run_manifest(
        checkpoint_dir,
        RunManifest(
            str(run.run_id),
            model.MODEL_NAME,
            model.MODEL_VERSION,
            run.config.max_epochs,
            run.config.learning_rate,
            run.config.batch_size,
            run.config.optimizer,
            structural_fingerprint(model.state_dict()),
        ),
    )
    return Checkpoint(str(target))

## 4. Train through the application handler

`handle_train_run` is the mlmodels write seam. We inject our adapter (curried on the
checkpoint dir + the training tensors) and hand it a `TrainRun` command carrying only
edge primitives. The handler certifies the id + knobs, drives the adapter, and returns a
`TrainedRun` aggregate pointing at the checkpoint on disk.

In [ ]:
cmd = TrainRun("ns-e2e", max_epochs=15, learning_rate=0.05, batch_size=64, optimizer="adam")
save = partial(real_save_trained_run, CKPT_DIR, X_train, Y_train)
trained = handle_train_run(save_trained_run=save, cmd=cmd).unwrap()
trained

## 5. Read side -- `find_run_summary` / `handle_get_run_summary`

The torch-free read path: the query handler re-wraps the edge id and binds the
`FindRunSummaryFn`, which projects the manifest straight into a `RunSummaryDTO` (no ckpt
load). We confirm the model identity round-trips through the manifest.

In [ ]:
summary = handle_get_run_summary(
    find=partial(find_run_summary, CKPT_DIR), query=GetRunSummary("ns-e2e")
).unwrap()
print(summary)
assert summary.model_name == SimpleSurrogate.MODEL_NAME
print("identity round-trips:", summary.model_name, summary.model_version)

## 6. Load side -- `load_trained_run`

The write-side LOAD counterpart: it reads the same manifest, **re-certifies** the
primitives back through the domain smart constructors (the trust boundary), and
reassembles a `TrainedRun`. Still torch-free -- the live nn is rebuilt separately.

In [ ]:
loaded = load_trained_run(CKPT_DIR, "ns-e2e").unwrap()
print(loaded)
assert loaded.config == trained.config
print("config round-trips:", loaded.config == trained.config)

## 7. Materialize the live surrogate and predict

`materialize_model` rebuilds the live nn from the injected factory, loads the ckpt
weights, and verifies the rebuilt model against the manifest on **both** axes
(structural fingerprint + declared identity). Then the surrogate predicts on the
held-out test slice -- de-standardized back to solar masses -- and we report the RMSE.
This is the end-to-end value: a trained surrogate that predicts, recovered purely from
what was persisted.

In [ ]:
live = materialize_model(
    lambda: build_model(loaded.config), IDENTITY, CKPT_DIR, loaded
).unwrap()
live.eval()
with torch.no_grad():
    pred_test = live(X_test) * y_std + y_mean
    truth_test = Y_test * y_std + y_mean
    rmse = torch.sqrt(nn.functional.mse_loss(pred_test, truth_test)).item()

print("first 5 predicted M (Msun):", [round(v, 3) for v in pred_test[:5].squeeze().tolist()])
print("first 5 true      M (Msun):", [round(v, 3) for v in truth_test[:5].squeeze().tolist()])
print(f"held-out test RMSE: {rmse:.4f} Msun")

## 8. BC-sync diagnostics -- deliberately probe the fault lines

The three cells below are the point of the spike: prove the reload guard fires, and
expose two real mismatches L2c must resolve.

### 8a. Reload guard fires on a drifted identity (expected: `MODEL_IDENTITY_MISMATCH`)

Materializing with a bumped version must fail loudly rather than load weights into a
model whose declared identity drifted. **This is the guard working correctly.**

In [ ]:
wrong = make_model_identity(SimpleSurrogate.MODEL_NAME, "9.9.9").unwrap()
res = materialize_model(lambda: build_model(loaded.config), wrong, CKPT_DIR, loaded)
print("is Err:", res.is_err())
print("cause:", res.unwrap_err())

### 8b. RESOLVED: `materialize_model` now loads a bundled checkpoint

The design's fork-2 persists via `trainer.save_checkpoint` -- a **bundled** Lightning
checkpoint (`state_dict` nested under a top-level dict with `epoch`, `optimizer_states`,
...). This once broke `materialize_model`, whose loader assumed a **bare** state_dict.
It is now fixed in `mlmodels` src: `_load_live_model` reads
`checkpoint.get("state_dict", checkpoint)`, so it loads bundled and bare checkpoints
alike. We save a bundle to a separate run id and materialize it -- it now succeeds.

*(This cell was the diagnostic that drove the first `src` harvest; it now proves the fix
rather than the bug.)*

In [ ]:
BUNDLE_DIR = CKPT_DIR / "bundle-probe"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
m2 = build_model(loaded.config)
tr2 = lightning.Trainer(
    max_epochs=1, accelerator="cpu", devices=1, logger=False,
    enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
)
tr2.fit(m2, DataLoader(TensorDataset(X_train, Y_train), batch_size=64))
tr2.save_checkpoint(BUNDLE_DIR / "bundle.ckpt")  # BUNDLED (state_dict nested + optimizer/epoch)
write_run_manifest(BUNDLE_DIR, RunManifest(
    "bundle", m2.MODEL_NAME, m2.MODEL_VERSION,
    loaded.config.max_epochs, loaded.config.learning_rate, loaded.config.batch_size,
    loaded.config.optimizer, structural_fingerprint(m2.state_dict())))

bundle_run = load_trained_run(BUNDLE_DIR, "bundle").unwrap()
res_bundle = materialize_model(lambda: build_model(loaded.config), IDENTITY, BUNDLE_DIR, bundle_run)
print("materialize on a bundled ckpt is Ok:", res_bundle.is_ok())
print("materialized model:", type(res_bundle.unwrap()).__name__)

### 8c. GAP: the shipped `train_run` facade is untrained and single-feature

The composition-root facade hardwires the untrained `1->1 SurrogateRegressor` -- it does
no fitting and cannot consume the 2-feature data. It writes a valid checkpoint + manifest
(so the persistence spine is exercised), but the identity is `surrogate-regressor` and the
only parameters are a `1x1` linear layer. **This is the L2c gap:** training a real
surrogate today means bypassing the facade and injecting an adapter into
`handle_train_run` (as this notebook does).

In [ ]:
facade_path = train_run(
    TrainRun("shipped", max_epochs=3, learning_rate=0.01, batch_size=8, optimizer="sgd")
)
man = read_run_manifest(CKPT_DIR, "shipped")
sd = torch.load(facade_path, weights_only=True)
print("facade identity:", man.model_name, man.model_version)
print("facade params:", {k: tuple(v.shape) for k, v in sd.items()})

## Findings

| # | Seam probed | Result |
|---|---|---|
| 1-7 | datasets load -> inject-train -> find -> load -> materialize -> predict | **PASS** -- the two BCs compose; a real trained surrogate is recovered from disk and predicts (test RMSE reported). |
| 8a | reload guard on identity drift | **PASS** -- `MODEL_IDENTITY_MISMATCH`, fails loudly. |
| 8b | fork-2 `trainer.save_checkpoint` vs `materialize_model` | **RESOLVED** -- `materialize_model` now reads `checkpoint.get("state_dict", ...)`, loading bundled + bare ckpts (first `src` harvest). |
| 8c | shipped `train_run` facade | **GAP** -- untrained `1->1` model; real training needs the injected-adapter seam. |

**Direction signal.** The persistence spine (L2b) holds up end-to-end with a real model,
and the one reload gap (8b) is now fixed in `src`. The missing value -- fit over real
data, a reported metric, a live prediction -- is exactly what these spikes wire by hand;
harvesting each proven-general piece into its owning BC is the ongoing path (8c, an
injected-model composition seat, is the next candidate).